In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# 1. CONFIGURAÇÕES
# ============================================================

fs = 1000                  # Frequência de amostragem (Hz)
N = 256                    # Número de amostras
f0 = 123.4                  # Frequência do sinal (Hz)
A = 1.0                     # Amplitude
phi = 0.0                   # Fase inicial

# Resolução original da DFT
delta_f = fs / N

# Zero padding para melhorar a visualização dos espectros
N_fft = 16 * N

# Vetor de tempo
n = np.arange(N)
t = n / 

# Geração da senoide
x = A * np.cos(2 * np.pi * f0 * t + phi)


# ============================================================
# 2. DEFINIÇÃO DAS JANELAS
# ============================================================

janelas = {
    "Retangular": np.ones(N),
    "Hann": np.hanning(N),
    "Hamming": np.hamming(N)
}


# ============================================================
# 3. FUNÇÃO PARA CALCULAR O ESPECTRO
# ============================================================

def calcular_espectro(sinal, janela):

    # Aplicação da janela
    sinal_janelado = sinal * janela

    # Cálculo da FFT
    X = np.fft.rfft(sinal_janelado, n=N_fft)

    # Eixo de frequências
    frequencias = np.fft.rfftfreq(
        N_fft,
        d=1/fs
    )

    # Compensação do ganho coerente da janela
    ganho = np.sum(janela)

    magnitude = np.abs(X) / ganho

    # Correção do espectro unilateral
    magnitude[1:-1] *= 2

    # Conversão para decibéis
    magnitude_db = 20 * np.log10(
        np.maximum(magnitude, 1e-12)
    )

    return sinal_janelado, frequencias, magnitude, magnitude_db


# ============================================================
# 4. CÁLCULO DOS ESPECTROS
# ============================================================

resultados = {}

for nome, janela in janelas.items():

    sinal_janelado, frequencias, magnitude, magnitude_db = (
        calcular_espectro(x, janela)
    )

    resultados[nome] = {
        "janela": janela,
        "sinal": sinal_janelado,
        "frequencias": frequencias,
        "magnitude": magnitude,
        "magnitude_db": magnitude_db
    }


# ============================================================
# 5. VISUALIZAÇÃO DAS JANELAS
# ============================================================

plt.figure(figsize=(11, 5))

for nome, janela in janelas.items():

    plt.plot(
        n,
        janela,
        label=nome
    )

plt.title("Comparação das funções janela")
plt.xlabel("Índice da amostra")
plt.ylabel("Amplitude da janela")

plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


# ============================================================
# 6. REPRESENTAÇÃO DO SINAL NO DOMÍNIO DO TEMPO
# ============================================================

fig, axes = plt.subplots(
    3, 1,
    figsize=(11, 9)
)

for ax, (nome, resultado) in zip(
    axes, resultados.items()
):

    ax.plot(
        n,
        x,
        label="Sinal original",
        color="blue"
    )

    ax.plot(
        n,
        resultado["sinal"],
        label="Sinal após aplicação da janela",
        color="red"
    )

    ax.set_title(nome)
    ax.set_xlabel("Índice da amostra")
    ax.set_ylabel("Amplitude")

    ax.legend()
    ax.grid(True)

plt.tight_layout()
plt.show()


# ============================================================
# 7. COMPARAÇÃO DOS ESPECTROS DE MAGNITUDE
# ============================================================

plt.figure(figsize=(12, 6))

for nome, resultado in resultados.items():

    plt.plot(
        resultado["frequencias"],
        resultado["magnitude"],
        label=nome
    )

plt.axvline(
    f0,
    color="black",
    linestyle="--",
    label=f"Frequência original ({f0} Hz)"
)

plt.xlim(70, 180)

plt.title("Comparação dos espectros de magnitude")
plt.xlabel("Frequência (Hz)")
plt.ylabel("Magnitude")

plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


# ============================================================
# 8. COMPARAÇÃO DOS ESPECTROS EM DECIBÉIS
# ============================================================

plt.figure(figsize=(12, 6))

for nome, resultado in resultados.items():

    # Normalização pelo pico de cada espectro
    espectro_db = (
        resultado["magnitude_db"]
- np.max(resultado["magnitude_db"])
    )

    plt.plot(
        resultado["frequencias"],
        espectro_db,
        label=nome
    )

plt.axvline(
    f0,
    color="black",
    linestyle="--",
    label="Frequência original"
)

plt.xlim(70, 180)
plt.ylim(-100, 5)

plt.title("Comparação dos espectros em decibéis")
plt.xlabel("Frequência (Hz)")
plt.ylabel("Magnitude relativa (dB)")

plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


# ============================================================
# 9. ANÁLISE DOS LÓBULOS DAS JANELAS
# ============================================================

def analisar_janela(janela):

    # FFT da própria janela com alta resolução
    W = np.fft.fftshift(
        np.abs(np.fft.fft(janela, n=131072))
    )

    eixo = np.fft.fftshift(
        np.fft.fftfreq(131072, d=1/fs)
    )

    centro = len(W) // 2

    # Procura o primeiro mínimo após o centro
    minimo = None

    for i in range(centro + 1, len(W) - 1):

        if W[i] <= W[i-1] and W[i] < W[i+1]:
            minimo = i
            break

    if minimo is None:
        return None, None

    # Largura do lóbulo principal
    largura = 2 * abs(eixo[minimo])

    # Maior lóbulo lateral
    pico_principal = W[centro]

    lobo_lateral = np.max(W[minimo:])

    nivel_lateral = 20 * np.log10(
        lobo_lateral / pico_principal
    )

    return largura, nivel_lateral


# ============================================================
# 10. RELATÓRIO COMPARATIVO
# ============================================================

print("\nANÁLISE DE JANELAMENTO E VAZAMENTO ESPECTRAL")
print("=" * 85)

print(f"Frequência do sinal: {f0} Hz")
print(f"Frequência de amostragem: {fs} Hz")
print(f"Número de amostras: {N}")
print(f"Resolução da DFT: {delta_f:.4f} Hz")

print("\nCARACTERÍSTICAS DAS JANELAS")
print("-" * 85)

print(
    f"{'Janela':<15}"
    f"{'Ganho coerente':<20}"
    f"{'Largura principal (Hz)':<25}"
    f"{'Lóbulo lateral (dB)':<20}"
)

for nome, janela in janelas.items():

    ganho = np.sum(janela) / N

    largura, nivel_lateral = analisar_janela(janela)

    print(
        f"{nome:<15}"
        f"{ganho:<20.4f}"
        f"{largura:<25.4f}"
        f"{nivel_lateral:<20.2f}"
    )


# ============================================================
# 11. IDENTIFICAÇÃO DA FREQUÊNCIA
# ============================================================

print("\nFREQUÊNCIAS IDENTIFICADAS")
print("-" * 50)

for nome, resultado in resultados.items():

    indice = np.argmax(resultado["magnitude"])

    frequencia_pico = resultado["frequencias"][indice]

    print(
        f"{nome:<15}: {frequencia_pico:.4f} Hz"
    )

print("\nSimulação concluída!")